In [ ]:
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
import pytz

In [ ]:
# Load the datasets
apps_df = pd.read_csv('Play Store Data.csv')
reviews_df = pd.read_csv('User Reviews.csv')

apps_df.head(3)

In [ ]:
reviews_df.head(3)

In [ ]:
# Drop missing values
apps_df = apps_df.dropna(subset=['App', 'Category', 'Rating', 'Reviews', 'Installs', 'Size']).copy()

# Clean Installs
apps_df['Installs'] = (
    apps_df['Installs']
    .astype(str)
    .str.replace('+', '', regex=False)
    .str.replace(',', '', regex=False)
)

apps_df = apps_df[apps_df['Installs'].str.isnumeric()].copy()
apps_df['Installs'] = apps_df['Installs'].astype(int)

# Clean Reviews and Rating
apps_df['Reviews'] = pd.to_numeric(apps_df['Reviews'], errors='coerce')
apps_df['Rating'] = pd.to_numeric(apps_df['Rating'], errors='coerce')

apps_df = apps_df.dropna(subset=['Reviews', 'Rating']).copy()
apps_df = apps_df[apps_df['Rating'] <= 5].copy()

In [ ]:
# Convert app size to MB
def parse_size_to_mb(val):
  val = str(val).strip()

  if val.endswith('M') or val.endswith('m'):
    try:
      return float(val[:-1])
    except:
      return None

  elif val.endswith('k') or val.endswith('K'):
    try:
      return float(val[:-1]) / 1024.0
    except:
      return None

  return None

apps_df['Size_MB'] = apps_df['Size'].apply(parse_size_to_mb)
apps_df = apps_df.dropna(subset=['Size_MB']).copy()

In [ ]:
# Clean Sentiment Subjectivity
reviews_df['Sentiment_Subjectivity'] = pd.to_numeric(reviews_df['Sentiment_Subjectivity'], errors='coerce')
reviews_df = reviews_df.dropna(subset=['App', 'Sentiment_Subjectivity']).copy()

# Calculate average subjectivity for each app
avg_subjectivity_df = reviews_df.groupby('App')['Sentiment_Subjectivity'].mean().reset_index()
avg_subjectivity_df.rename(columns={'Sentiment_Subjectivity': 'Avg_Subjectivity'}, inplace=True)

# Merge Play Store data with average subjectivity
merged_df = pd.merge(apps_df, avg_subjectivity_df, on='App', how='inner')
merged_df.head(3)

In [ ]:
# Required categories
target_categories = ['GAME', 'BEAUTY', 'BUSINESS', 'COMICS', 'COMMUNICATION', 'DATING', 'ENTERTAINMENT', 'SOCIAL', 'EVENTS']

# Filter conditions
cond_rating = merged_df['Rating'] > 3.5
cond_category = merged_df['Category'].str.upper().isin(target_categories)
cond_reviews = merged_df['Reviews'] > 500
cond_installs = merged_df['Installs'] > 50000
cond_subjectivity = merged_df['Avg_Subjectivity'] > 0.5
cond_no_s = ~merged_df['App'].astype(str).str.contains('S', case=True, na=False)

filtered_df = merged_df[
    cond_rating
    & cond_category
    & cond_reviews
    & cond_installs
    & cond_subjectivity
    & cond_no_s
].copy()

print(f'Total filtered rows: {len(filtered_df)}')
filtered_df[['App', 'Category', 'Rating', 'Reviews', 'Installs', 'Size_MB', 'Avg_Subjectivity']].head(10)

In [ ]:
# Translate required category names
category_translation_map = {
    'BEAUTY': 'सुंदरता',
    'BUSINESS': 'வணிகம்',
    'DATING': 'Partnersuche'
}

def get_display_category(category):
  cat_upper = str(category).upper()
  return category_translation_map.get(cat_upper, cat_upper.title())

filtered_df['Category_Display'] = filtered_df['Category'].apply(get_display_category)

In [ ]:
# Check time (5:00 PM to 7:00 PM IST)
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()

start_time = datetime.strptime('17:00:00', '%H:%M:%S').time()
end_time = datetime.strptime('19:00:00', '%H:%M:%S').time()

is_within_time = start_time <= current_time < end_time

graph_path = 'task_5_graph.html'
dashboard_path = 'task_5_dashboard.html'

In [ ]:
# Category summary for dashboard
category_summary = filtered_df.groupby('Category_Display').agg(
    App_Count=('App', 'count'),
    Average_Rating=('Rating', 'mean'),
    Total_Installs=('Installs', 'sum')
).reset_index()

cat_rows_html = ''
for idx, row in category_summary.iterrows():
  cat_rows_html += f"<tr><td>{row['Category_Display']}</td><td>{int(row['App_Count'])}</td><td>{row['Average_Rating']:.2f}</td><td>{int(row['Total_Installs']):,}</td></tr>\n"

In [ ]:
# Generating the graph using Plotly and saving it as task_5_graph.html silently if within the allowed time window
if is_within_time:
  fig = go.Figure()
  sizeref = 2.0 * filtered_df['Installs'].max() / (45 ** 2)

  for category in filtered_df['Category'].drop_duplicates():
    category_df = filtered_df[filtered_df['Category'] == category]
    category_display = get_display_category(category)
    marker_color = 'pink' if category == 'GAME' else None

    fig.add_trace(
        go.Scatter(
            x=category_df['Size_MB'],
            y=category_df['Rating'],
            mode='markers',
            name=category_display,
            text=category_df['App'],
            customdata=category_df[['Installs', 'Reviews', 'Avg_Subjectivity']],
            marker=dict(size=category_df['Installs'], sizemode='area', sizeref=sizeref, sizemin=6, color=marker_color, opacity=0.75, line=dict(width=1, color='DarkSlateGrey')),
            hovertemplate=(
                '<b>%{text}</b><br>'
                f'Category: {category_display}<br>'
                'Size: %{x:.2f} MB<br>'
                'Rating: %{y:.2f}<br>'
                'Installs: %{customdata[0]:,}<br>'
                'Reviews: %{customdata[1]:,}<br>'
                'Average Subjectivity: %{customdata[2]:.2f}<br>'
                '<extra></extra>'
            )
        )
    )

  fig.update_layout(
      title=dict(text='App Size vs Average Rating<br><sup>Bubble Size = Installs | Game Category Highlighted in Pink</sup>', x=0.5, xanchor='center'),
      xaxis=dict(title='App Size (MB)', showgrid=True),
      yaxis=dict(title='Average Rating', range=[3.5, 5], showgrid=True),
      legend=dict(title='Categories', orientation='v', x=1.02, y=1, xanchor='left', yanchor='top'),
      template='plotly_white',
      hovermode='closest',
      height=700,
      margin=dict(l=70, r=180, t=110, b=70)
  )

  fig.write_html(graph_path)

  status_section_html = f"""
    <div class="card">
        <h2>Graph Status</h2>
        <p style="color: green; font-weight: bold;">Status: Graph Successfully Generated</p>
        <p>The graph was generated because the current time is between 5 PM and 7 PM IST.</p>
        <p><a href="{graph_path}" target="_blank" style="display:inline-block; padding:10px 15px; background-color:#1a73e8; color:white; text-decoration:none; border-radius:5px;">Open Graph</a></p>
    </div>
    """ 
else:
  status_section_html = """
  <div class="card">
      <h2>Graph Status</h2>
      <p style="color: red; font-weight: bold;">Status: Graph Restricted</p>
      <p>The graph is available only between 5 PM and 7 PM IST.</p>
  </div>
  """

In [ ]:
# Generate dashboard
dashboard_html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Play Store App Analysis Dashboard - Task 5</title>
    <style>
        body {{font-family: Arial, sans-serif; margin: 30px; background-color: #f4f6f9; color: #333;}}
        h1 {{color: #1a73e8;}}
        .card {{background: white; padding: 20px; margin-bottom: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);}}
        table {{width: 100%; border-collapse: collapse; margin-top: 10px;}}
        th, td {{padding: 10px; border: 1px solid #ddd; text-align: left;}}
        th {{background-color: #f0f4f8;}}
    </style>
</head>
<body>
    <h1>Play Store App Analysis Dashboard - Task 5</h1>
    <div class="card">
        <h2>Dashboard Summary</h2>
        <p><b>Filters Applied:</b> Rating &gt; 3.5, Reviews &gt; 500, Installs &gt; 50,000, Average Sentiment Subjectivity &gt; 0.5, selected categories only, and app name should not contain uppercase 'S'.</p>
        <p><b>Total Filtered Rows:</b> {len(filtered_df):,}</p>
    </div>
    <div class="card">
        <h2>Category Summary</h2>
        <table>
            <tr><th>Category</th><th>Apps</th><th>Average Rating</th><th>Total Installs</th></tr>
            {cat_rows_html}
        </table>
    </div>
    {status_section_html}
</body>
</html>
"""

with open(dashboard_path, 'w', encoding='utf-8') as f:
  f.write(dashboard_html_content)